# Confounder Adjustment — African-American Cohort (GSE148375)

**Purpose:** Build the 12-dimensional confounder block (10 genetic PCs + age + gender)
for DoubleML residualization, using relatedness-filtered data (Step 8 of 01_qc.ipynb).

**Inputs:**
- `checkpoint7b_snp_encoded_012_relatedness_filtered.csv`
- `checkpoint2b_metadata_relatedness_filtered.csv`
- `HumanExome-12-v1-0-B.csv` (manifest)

**Method:** Exclude sex-linked probes → exclude monomorphic SNPs → EIGENSTRAT
standardize (native float64 throughout — see Step 6 note below) → PCA
(`svd_solver='full'`, exact) → assemble with standardized age + binary gender.

**Outputs:** `confounders_X.npy`, `pca_diagnostics.csv`,
`pc1_gender_admixture_summary.csv`, plus a mandatory genomic-inflation validation
(Step 7) that must pass/be documented before proceeding to 03_snp_selection.

## Step 1 — Load relatedness-filtered genotype matrix, exclude sex-linked probes

In [1]:
import pandas as pd
import numpy as np
import re
import os
import gc

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_address_suffix(probe_id):
    return re.sub(r'_\d+$', '', probe_id)

encoded_df = pd.read_csv(os.path.join(out_dir, "checkpoint7b_snp_encoded_012_relatedness_filtered.csv"))
probe_id_array = encoded_df["probe_id"].to_numpy()
sample_cols = encoded_df.columns[1:]
sample_ids = sample_cols.tolist()
X_snp_first = encoded_df[sample_cols].to_numpy(dtype=np.int8)
del encoded_df
gc.collect()

print("Loaded matrix shape (SNPs x samples):", X_snp_first.shape)

manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_probes = set(manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "IlmnID"])
sex_linked_core_names = {strip_address_suffix(p) for p in sex_linked_probes}
our_probe_core_names = {strip_address_suffix(p): p for p in probe_id_array}
to_exclude = {our_probe_core_names[c] for c in sex_linked_core_names if c in our_probe_core_names}
del manifest_df
gc.collect()

print("Sex-linked probes to exclude:", len(to_exclude))

Loaded matrix shape (SNPs x samples): (238927, 3036)
Sex-linked probes to exclude: 5317


## Step 2 — Autosomal filter, monomorphic exclusion, EIGENSTRAT standardization
Native float64 throughout (a prior version of this pipeline cast an already-lossy
float32 standardized matrix to float64 *after* the division step, which preserved
corrupted values — see project history. This version avoids that by never touching
float32 for the standardized matrix itself).

In [2]:
import numpy as np
import gc

keep_mask = ~np.isin(probe_id_array, list(to_exclude))
X_auto_int = X_snp_first[keep_mask]
del X_snp_first
gc.collect()

X_auto_64 = X_auto_int.T.astype(np.float64)
del X_auto_int
gc.collect()

print("X_auto shape (samples x SNPs, autosomal):", X_auto_64.shape)

p = X_auto_64.mean(axis=0) / 2
denom = np.sqrt(2 * p * (1 - p))
valid_snp_mask = denom > 1e-8
print("Monomorphic/invalid SNPs excluded:", (~valid_snp_mask).sum())

X_standardized = (X_auto_64[:, valid_snp_mask] - 2 * p[valid_snp_mask]) / denom[valid_snp_mask]
del X_auto_64
gc.collect()

print("Standardized matrix shape:", X_standardized.shape)
print("Mean (should be ~0):", X_standardized.mean())
print("Std (should be ~1):", X_standardized.std())

probe_ids_after_autosomal = probe_id_array[keep_mask]
probe_ids_valid = probe_ids_after_autosomal[valid_snp_mask]
print("Final informative SNP count:", len(probe_ids_valid))

X_auto shape (samples x SNPs, autosomal): (3036, 233610)
Monomorphic/invalid SNPs excluded: 92286
Standardized matrix shape: (3036, 141324)
Mean (should be ~0): 1.161799390847652e-18
Std (should be ~1): 1.007551515838158
Final informative SNP count: 141324


## Step 3 — PCA (exact, full SVD)

In [3]:
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

pca_std = PCA(n_components=10, random_state=42, svd_solver='full')
pcs_std = pca_std.fit_transform(X_standardized)

print("PC scores shape:", pcs_std.shape)
print("Explained variance ratio per PC:", pca_std.explained_variance_ratio_)
print("Cumulative variance explained:", np.cumsum(pca_std.explained_variance_ratio_))

pca_diag = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(10)],
    "explained_variance_ratio": pca_std.explained_variance_ratio_,
    "cumulative_variance": np.cumsum(pca_std.explained_variance_ratio_)
})
pca_diag.to_csv(os.path.join(out_dir, "pca_diagnostics.csv"), index=False)
print("Saved.")

PC scores shape: (3036, 10)
Explained variance ratio per PC: [0.00595154 0.00116491 0.00100857 0.00092944 0.0008682  0.00085298
 0.00082531 0.00080676 0.00079536 0.00078156]
Cumulative variance explained: [0.00595154 0.00711645 0.00812502 0.00905446 0.00992266 0.01077564
 0.01160095 0.01240771 0.01320307 0.01398463]
Saved.


## Step 4 — Confounder block assembly (10 PCs + age + gender)

In [4]:
import pandas as pd
import numpy as np
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

meta_df = pd.read_csv(os.path.join(out_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
meta_df["sample_id"] = meta_df["sample_id"].astype(str)

meta_aligned = meta_df.set_index("sample_id").reindex(sample_ids).reset_index()

age_std = ((meta_aligned["age"].astype(float) - meta_aligned["age"].astype(float).mean()) /
           meta_aligned["age"].astype(float).std()).values
gender_binary = (meta_aligned["gender"] == "Male").astype(np.float64).values

X_confounders = np.hstack([
    pcs_std,
    age_std.reshape(-1, 1),
    gender_binary.reshape(-1, 1)
]).astype(np.float64)

print("Confounder block shape:", X_confounders.shape)

np.save(os.path.join(out_dir, "confounders_X.npy"), X_confounders)
print("Saved.")

Confounder block shape: (3036, 12)
Saved.


## Step 5 — PC1-gender documentation (admixture finding)
Documented cross-cohort finding: PC1 correlates strongly with self-reported gender
even after excluding sex chromosomes. Only 1-2 individual autosomal SNPs drive this
at the SNP level (checked below); the bulk of PC1's gender association is diffuse
across many chromosomes, consistent with sex-biased admixture rather than residual
sex-chromosome contamination. Replicated independently in the EA cohort at nearly
identical magnitude — see EA's 02_confounders.ipynb for comparison and discussion
of what that replication implies (admixture alone may not fully explain it, since EA
does not share AA's admixture history — see PPP1R12B probe-level discussion in
04_pc_algorithm notebooks).

In [5]:
import pandas as pd
import numpy as np
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

pc_df = pd.DataFrame(pcs_std, columns=[f"PC{i+1}" for i in range(10)])
pc_df["sample_id"] = sample_ids
merged_check = meta_df[["sample_id", "age", "gender", "smoking_status"]].merge(pc_df, on="sample_id", how="inner")

print("PC1 mean by gender:")
print(merged_check.groupby("gender")["PC1"].mean())
print("\nPC1 mean by smoking_status:")
print(merged_check.groupby("smoking_status")["PC1"].mean())

gender_map = meta_df.set_index("sample_id")["gender"]
gender_aligned = np.array([gender_map.get(sid, None) for sid in sample_ids])
gender_binary_check = (gender_aligned == "Male").astype(int)

X_centered = X_standardized - X_standardized.mean(axis=0)
gender_centered = gender_binary_check - gender_binary_check.mean()
cov = (X_centered * gender_centered[:, None]).mean(axis=0)
snp_std = X_standardized.std(axis=0)
corr_with_gender = cov / (snp_std * gender_binary_check.std() + 1e-9)

n_strong_03 = (np.abs(corr_with_gender) > 0.3).sum()
print(f"\nSNPs with |corr with gender| > 0.3: {n_strong_03}")
print(f"Max individual SNP-gender correlation: {np.abs(corr_with_gender).max():.6f}")

pc1_gender_summary = pd.DataFrame({
    "metric": ["PC1_female_mean", "PC1_male_mean", "n_snps_corr_gt_0.3", "max_snp_gender_corr"],
    "value": [
        merged_check.loc[merged_check["gender"]=="Female", "PC1"].mean(),
        merged_check.loc[merged_check["gender"]=="Male", "PC1"].mean(),
        n_strong_03, np.abs(corr_with_gender).max()
    ]
})
pc1_gender_summary.to_csv(os.path.join(out_dir, "pc1_gender_admixture_summary.csv"), index=False)
print("\nSaved.")

PC1 mean by gender:
gender
Female   -1.197838
Male      1.370408
Name: PC1, dtype: float64

PC1 mean by smoking_status:
smoking_status
Non-smoker   -0.652973
Smoker        0.705783
Name: PC1, dtype: float64

SNPs with |corr with gender| > 0.3: 2
Max individual SNP-gender correlation: 0.859923

Saved.


## Step 6 — VALIDATION GATE: Genomic inflation factor (λ)
This is a mandatory check before proceeding to 03_snp_selection. λ close to 1.0
indicates well-calibrated confounder adjustment; λ > 1.1 indicates residual
confounding/non-independence not captured by the current confounder block.

**Result:** λ = 1.74 (mean of 5 seeds, ±0.25) — see below.

**Interpretation:** Despite relatedness filtering (Step 8 of 01_qc.ipynb) and
exact-SVD PCA, meaningful residual inflation remains. Sweeping PC count (10→40)
was tested and made inflation *worse*, ruling out "insufficient PCs" as the fix.
Root cause not fully resolved — documented as an open limitation of this analysis,
consistent with the same unresolved pattern found in the EA cohort. Downstream
stability selection thresholds (03_snp_selection) should be interpreted with this
caveat: a dual primary/sensitivity threshold approach (80% / 100% stability) is
used throughout this project specifically to hedge against this known limitation.

In [6]:
import numpy as np
from sklearn.model_selection import KFold
from scipy import stats
import time

def doubleml_scan(X_snps, Y, X_conf, n_folds=5, random_state=42):
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    D_resid = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)
    Xc = np.column_stack([np.ones(n), X_conf])

    for train_idx, test_idx in kf.split(Xc):
        Xc_tr, Xc_te = Xc[train_idx], Xc[test_idx]
        coef_Y = np.linalg.lstsq(Xc_tr, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_te @ coef_Y
        coef_D = np.linalg.lstsq(Xc_tr, X_snps[train_idx], rcond=None)[0]
        D_resid[test_idx] = X_snps[test_idx] - Xc_te @ coef_D

    Yr = Y_resid - Y_resid.mean()
    Dr = D_resid - D_resid.mean(axis=0)
    del D_resid

    sum_DY = (Dr * Yr[:, None]).sum(axis=0)
    sum_DD = (Dr ** 2).sum(axis=0)
    sum_YY = (Yr ** 2).sum()

    theta = sum_DY / sum_DD
    ssr = sum_YY - (sum_DY ** 2) / sum_DD
    sigma2 = ssr / (n - 2)
    se = np.sqrt(sigma2 / sum_DD)
    t_stat = theta / se
    pvals = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))
    return theta, pvals

def compute_lambda(pvals):
    chi2_observed = stats.chi2.isf(pvals, df=1)
    return np.median(chi2_observed) / stats.chi2.ppf(0.5, df=1)

status_map = meta_df.set_index("sample_id")["smoking_status"]
Y = np.array([1.0 if status_map.get(sid) == "Smoker" else 0.0 for sid in sample_ids])

print("Running 5 seeds for stable lambda estimate...")
lambdas = []
for seed in range(5):
    _, pv = doubleml_scan(X_standardized, Y, X_confounders, random_state=seed)
    lam = compute_lambda(pv)
    lambdas.append(lam)
    print(f"seed={seed}: lambda={lam:.4f}")

mean_lambda = np.mean(lambdas)
print(f"\nMean lambda: {mean_lambda:.4f} (+/- {np.std(lambdas):.4f})")
print(f"GATE STATUS: {'PASS' if mean_lambda < 1.1 else 'FAIL — documented limitation, proceed with dual-threshold approach'}")

Running 5 seeds for stable lambda estimate...
seed=0: lambda=1.8431
seed=1: lambda=1.5341
seed=2: lambda=1.3930
seed=3: lambda=2.0895
seed=4: lambda=1.8507

Mean lambda: 1.7421 (+/- 0.2481)
GATE STATUS: FAIL — documented limitation, proceed with dual-threshold approach
